<a href="https://colab.research.google.com/github/uppikaur/aml/blob/master/LabAssignment1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

In [ ]:
X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])

y = np.array([[0], [1], [1], [0]])

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    return x * (1 - x)

def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

In [ ]:
class MLP:
    def __init__(self, input_size, hidden_layers, output_size, lr=0.1, activation='sigmoid'):
        self.lr = lr
        self.activation_name = activation

        # Architecture
        layers = [input_size] + hidden_layers + [output_size]

        # Initialize weights & biases
        self.weights = []
        self.biases = []

        for i in range(len(layers) - 1):
            self.weights.append(np.random.randn(layers[i], layers[i+1]))
            self.biases.append(np.zeros((1, layers[i+1])))

    def activation(self, x):
        if self.activation_name == 'sigmoid':
            return sigmoid(x)
        return relu(x)

    def activation_derivative(self, x):
        if self.activation_name == 'sigmoid':
            return sigmoid_derivative(x)
        return relu_derivative(x)

    # Forward Propagation
    def forward(self, X):
        self.activations = [X]

        for w, b in zip(self.weights, self.biases):
            z = np.dot(self.activations[-1], w) + b
            a = self.activation(z)
            self.activations.append(a)

        return self.activations[-1]

    # Loss (MSE)
    def loss(self, y_true, y_pred):
        return np.mean((y_true - y_pred) ** 2)

    # Backpropagation
    def backward(self, y_true):
        m = y_true.shape[0]

        deltas = [None] * len(self.weights)

        # Output layer error
        error = self.activations[-1] - y_true
        deltas[-1] = error * self.activation_derivative(self.activations[-1])

        # Hidden layers
        for i in reversed(range(len(deltas) - 1)):
            error = np.dot(deltas[i+1], self.weights[i+1].T)
            deltas[i] = error * self.activation_derivative(self.activations[i+1])

        # Update weights
        for i in range(len(self.weights)):
            self.weights[i] -= self.lr * np.dot(self.activations[i].T, deltas[i]) / m
            self.biases[i] -= self.lr * np.sum(deltas[i], axis=0, keepdims=True) / m

    # Training
    def train(self, X, y, epochs=10000, print_every=1000):
        for epoch in range(epochs):
            y_pred = self.forward(X)
            loss = self.loss(y, y_pred)

            self.backward(y)

            if epoch % print_every == 0:
                print(f"Epoch {epoch}, Loss: {loss:.6f}")

In [ ]:
mlp = MLP(input_size=2, hidden_layers=[4], output_size=1, lr=0.1)
mlp.train(X, y, epochs=10000)

print("Predictions:")
print(mlp.forward(X))

Epoch 0, Loss: 0.257028
Epoch 1000, Loss: 0.248936
Epoch 2000, Loss: 0.247167
Epoch 3000, Loss: 0.244158
Epoch 4000, Loss: 0.238565
Epoch 5000, Loss: 0.228458
Epoch 6000, Loss: 0.211484
Epoch 7000, Loss: 0.186202
Epoch 8000, Loss: 0.153794
Epoch 9000, Loss: 0.117874
Predictions:
[[0.16443793]
 [0.70144078]
 [0.71541833]
 [0.37830115]]
